In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):

    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node(
    "add_one",
    add_one
)

builder.add_node(
    "add_two",
    add_two
)

builder.add_node(
    "add_three",
    add_three
)


# =========================================================
# 4. GRAPH FLOW
# =========================================================

builder.add_edge(
    START,
    "add_one"
)

builder.add_edge(
    "add_one",
    "add_two"
)

builder.add_edge(
    "add_two",
    "add_three"
)

builder.add_edge(
    "add_three",
    END
)


# =========================================================
# 5. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()


# =========================================================
# 6. COMPILE
# =========================================================

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 7. THREAD 001
# =========================================================

config_001 = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n========================================")
print("THREAD 001 EXECUTION")
print("========================================")


result_001 = graph.invoke(
    {
        "count": 0
    },
    config=config_001
)

print("\nThread 001 Final Result:")
print(result_001)


# =========================================================
# 8. THREAD 001 CURRENT STATE
# =========================================================

state_001 = graph.get_state(
    config_001
)

print("\nThread 001 Current State:")
print(
    "Values:",
    state_001.values
)

print(
    "Thread ID:",
    state_001.config["configurable"]["thread_id"]
)

print(
    "Checkpoint ID:",
    state_001.config["configurable"]["checkpoint_id"]
)

print(
    "Next Node:",
    state_001.next
)


# =========================================================
# 9. THREAD 002
# =========================================================

config_002 = {
    "configurable": {
        "thread_id": "thread-002"
    }
}


print("\n========================================")
print("THREAD 002 EXECUTION")
print("========================================")


result_002 = graph.invoke(
    {
        "count": 100
    },
    config=config_002
)

print("\nThread 002 Final Result:")
print(result_002)


# =========================================================
# 10. THREAD 002 CURRENT STATE
# =========================================================

state_002 = graph.get_state(
    config_002
)

print("\nThread 002 Current State:")

print(
    "Values:",
    state_002.values
)

print(
    "Thread ID:",
    state_002.config["configurable"]["thread_id"]
)

print(
    "Checkpoint ID:",
    state_002.config["configurable"]["checkpoint_id"]
)

print(
    "Next Node:",
    state_002.next
)


# =========================================================
# 11. COMPARE BOTH THREADS
# =========================================================

print("\n========================================")
print("THREAD COMPARISON")
print("========================================")

print(
    "Thread 001:",
    state_001.values
)

print(
    "Thread 002:",
    state_002.values
)


# =========================================================
# 12. THREAD 001 CHECKPOINT HISTORY
# =========================================================

print("\n========================================")
print("THREAD 001 CHECKPOINT HISTORY")
print("========================================")


history_001 = list(
    graph.get_state_history(
        config_001
    )
)


print(
    "Total Checkpoints:",
    len(history_001)
)


for state in history_001:

    checkpoint_id = (
        state.config["configurable"]["checkpoint_id"]
    )

    print("\n----------------------------------------")

    print(
        "Thread ID:",
        state.config["configurable"]["thread_id"]
    )

    print(
        "Checkpoint ID:",
        checkpoint_id
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next Node:",
        state.next
    )

    print(
        "Metadata:",
        state.metadata
    )


# =========================================================
# 13. SELECT AN OLD CHECKPOINT FROM THREAD 001
# =========================================================

if history_001:

    # get_state_history returns newest -> oldest
    old_checkpoint = history_001[-1]

    old_checkpoint_id = (
        old_checkpoint
        .config["configurable"]["checkpoint_id"]
    )

    old_checkpoint_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }


    # =====================================================
    # 14. READ OLD CHECKPOINT
    # =====================================================

    old_state = graph.get_state(
        old_checkpoint_config
    )


    print("\n========================================")
    print("SELECTED OLD CHECKPOINT")
    print("========================================")

    print(
        "Old Thread ID:",
        old_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Old Checkpoint ID:",
        old_checkpoint_id
    )

    print(
        "Old Values:",
        old_state.values
    )

    print(
        "Old Next Node:",
        old_state.next
    )


# =========================================================
# 15. SWITCH BACK TO THREAD 001
# =========================================================

print("\n========================================")
print("SWITCH BACK TO THREAD 001")
print("========================================")

again_state_001 = graph.get_state(
    config_001
)

print(
    "Thread:",
    again_state_001.config[
        "configurable"
    ]["thread_id"]
)

print(
    "Values:",
    again_state_001.values
)


# =========================================================
# 16. SWITCH TO THREAD 002
# =========================================================

print("\n========================================")
print("SWITCH TO THREAD 002")
print("========================================")

again_state_002 = graph.get_state(
    config_002
)

print(
    "Thread:",
    again_state_002.config[
        "configurable"
    ]["thread_id"]
)

print(
    "Values:",
    again_state_002.values
)